# 漫畫 OCR 擂台(ブラックジャックによろしく 第 1 卷):PaddleOCR-VL-1.5 vs manga-ocr vs Qwen3.8

1. **PaddleOCR-VL-1.5** 抓文字框並辨識(OmniDocBench 第二名,0.9B,框用專門的版面偵測模型)
2. 同一批框裁切後,交給 **manga-ocr**(日文漫畫專用)和 **Qwen3.8-27B**(可選,連你另一個 session 的 vLLM)各讀一次
3. 輸出 `report.html`:每個框的裁圖 + 三個結果並排,你直接比

**執行階段**:L4 就夠(Paddle 和 manga-ocr 都很小)。Qwen 走你 G4 session 的 tunnel,不用在這裡載 27B。

## 1. 設定

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/manga-ocr"   # 已建好這個資料夾
PDF_PATH  = f"{DRIVE_DIR}/001bj.pdf"              # 把 001bj.pdf 拖進上面那個資料夾
PAGES     = "7-26"     # 要測的頁碼(PDF 頁碼,從 1 起算);例:"7-26" 或 "7,9,12-15"
OUT_DIR   = f"{DRIVE_DIR}/out"                    # 結果直接存回 Drive,斷線也不會不見

# Qwen(選用):填另一個 session 印出的值;留空就只比 Paddle vs manga-ocr
QWEN_BASE_URL = ""   # 例:https://xxxx.trycloudflare.com/v1
QWEN_API_KEY  = ""
QWEN_MODEL    = "qwen3.8-27b"

MIN_BOX = 12         # 小於這個像素的框忽略

## 2. 安裝(約 5 分鐘)

Paddle 會換掉 Colab 內建 PyTorch 依賴的 NCCL/CUDA 函式庫,裝在同一個環境會讓 `import torch` 壞掉。所以 Paddle 放在獨立的 venv,用子行程執行;主環境只裝 PyTorch 系的套件。

In [ ]:
# 主環境:不碰 torch
!pip install -q manga-ocr pymupdf openai
import torch; print("torch", torch.__version__, "CUDA:", torch.cuda.is_available())

# Paddle 獨立環境(Paddle 會換掉 torch 依賴的 NCCL/CUDA 函式庫,不能裝在一起)
!pip install -q uv
!uv venv -q --seed /content/pdenv
!uv pip install -q --python /content/pdenv/bin/python paddlepaddle-gpu==3.2.1 --index-url https://www.paddlepaddle.org.cn/packages/stable/cu126/ --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
!uv pip install -q --python /content/pdenv/bin/python "paddleocr[doc-parser]"
!/content/pdenv/bin/python -c "import paddle; print('paddle', paddle.__version__, 'CUDA:', paddle.device.is_compiled_with_cuda())"

## 3. 從 Drive 的 PDF 抽出指定頁(取內嵌原圖,不重新壓縮)

In [ ]:
import os, glob
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
import pymupdf as fitz

def parse_pages(spec):
    out = []
    for part in spec.split(","):
        a, _, b = part.strip().partition("-")
        out += list(range(int(a), int(b or a) + 1))
    return out

assert os.path.exists(PDF_PATH), f"找不到 {PDF_PATH}:先把 PDF 放進 Drive 的 manga-ocr 資料夾"
os.makedirs(f"{OUT_DIR}/png", exist_ok=True)
doc = fitz.open(PDF_PATH)
pages = []
for n in parse_pages(PAGES):
    page = doc[n - 1]
    imgs = page.get_images(full=True)
    q = f"{OUT_DIR}/png/p{n:03d}"
    if len(imgs) == 1:                       # 掃描型 PDF:一頁一張圖,直接取原檔
        x = doc.extract_image(imgs[0][0]); q += "." + x["ext"]
        open(q, "wb").write(x["image"])
    else:                                    # 其他情況就渲染
        q += ".png"; page.get_pixmap(dpi=150).save(q)
    pages.append(q)
print(len(pages), "頁:", [Path(x).name for x in pages])

## 4. PaddleOCR-VL-1.5:偵測 + 辨識(在獨立 venv 執行)

In [ ]:
import json
SCRIPT = "import json, sys\nfrom paddleocr import PaddleOCRVL\n\npages, out_path, min_box = json.load(open(sys.argv[1])), sys.argv[2], int(sys.argv[3])\n\ndef walk_blocks(obj):\n    # 欄位名稱跨版本有差,所以寬鬆地找 (bbox, text, label)\n    if isinstance(obj, dict):\n        bbox = obj.get(\"block_bbox\") or obj.get(\"coordinate\") or obj.get(\"bbox\")\n        text = obj.get(\"block_content\") or obj.get(\"text\") or obj.get(\"rec_text\")\n        if bbox is not None and len(bbox) == 4 and isinstance(text, str):\n            yield [float(v) for v in bbox], text, obj.get(\"block_label\") or obj.get(\"label\") or \"\"\n        for v in obj.values():\n            yield from walk_blocks(v)\n    elif isinstance(obj, list):\n        for v in obj:\n            yield from walk_blocks(v)\n\npipe = PaddleOCRVL(pipeline_version=\"v1.5\")\nresults, raw = {}, {}\nfor p in pages:\n    blocks, seen = [], set()\n    for res in pipe.predict(p):\n        raw.setdefault(p, []).append(res.json)\n        for bbox, text, label in walk_blocks(res.json):\n            key = tuple(round(v) for v in bbox)\n            if key in seen or not text.strip():\n                continue\n            if bbox[2] - bbox[0] < min_box or bbox[3] - bbox[1] < min_box:\n                continue\n            seen.add(key)\n            blocks.append({\"bbox\": bbox, \"label\": label, \"paddle\": text.strip()})\n    results[p] = blocks\n    print(p.rsplit(\"/\", 1)[-1], len(blocks), \"boxes\", flush=True)\n\njson.dump(results, open(out_path, \"w\"), ensure_ascii=False)\njson.dump(raw, open(out_path + \".raw.json\", \"w\"), ensure_ascii=False, default=str)\n"
open('/content/paddle_run.py', 'w').write(SCRIPT)
json.dump(pages, open('/content/pages.json', 'w'))
!/content/pdenv/bin/python /content/paddle_run.py /content/pages.json /content/paddle_out.json {MIN_BOX}
results = json.load(open('/content/paddle_out.json'))
print('共', sum(len(v) for v in results.values()), '個框;原始輸出:/content/paddle_out.json.raw.json')

## 5. 同一批框:manga-ocr 與 Qwen 各讀一次

In [ ]:
import base64, io, json
from PIL import Image
from manga_ocr import MangaOcr
mocr = MangaOcr()

qwen = None
if QWEN_BASE_URL:
    from openai import OpenAI
    qwen = OpenAI(base_url=QWEN_BASE_URL, api_key=QWEN_API_KEY, timeout=120)

def to_b64(im):
    buf = io.BytesIO(); im.save(buf, "PNG"); return base64.b64encode(buf.getvalue()).decode()

def ask_qwen(crop):
    r = qwen.chat.completions.create(
        model=QWEN_MODEL, temperature=0,
        messages=[{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": "data:image/png;base64," + to_b64(crop)}},
            {"type": "text", "text": "逐字抄寫圖中的文字(直書由右到左讀)。只輸出文字本身,不要翻譯、不要解釋。"}]}],
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    return r.choices[0].message.content.strip()

os.makedirs(f"{OUT_DIR}/crops", exist_ok=True)
for p, blocks in results.items():
    im = Image.open(p)
    for i, b in enumerate(blocks):
        x1, y1, x2, y2 = b["bbox"]
        crop = im.crop((max(0, x1-4), max(0, y1-4), min(im.width, x2+4), min(im.height, y2+4)))
        b["crop"] = f"crops/{Path(p).stem}_{i:03d}.png"; crop.save(f"{OUT_DIR}/{b['crop']}")
        b["manga_ocr"] = mocr(crop)
        if qwen:
            try: b["qwen"] = ask_qwen(crop)
            except Exception as e: b["qwen"] = f"[錯誤] {e}"
    print(Path(p).name, "完成")
json.dump(results, open(f"{OUT_DIR}/results.json", "w"), ensure_ascii=False, indent=1)

## 6. 產生並排報告

In [ ]:
import html
rows = []
for p, blocks in results.items():
    rows.append(f"<tr><th colspan=5 class=pg>{html.escape(Path(p).name)}</th></tr>")
    for i, b in enumerate(blocks):
        img = base64.b64encode(open(f"{OUT_DIR}/{b['crop']}", "rb").read()).decode()
        cells = [b.get("paddle", ""), b.get("manga_ocr", ""), b.get("qwen", "—")]
        rows.append(f"<tr><td>{i+1}<br><small>{html.escape(b['label'])}</small></td>"
                    f"<td><img src='data:image/png;base64,{img}'></td>"
                    + "".join(f"<td>{html.escape(c)}</td>" for c in cells) + "</tr>")
page = f"""<!doctype html><meta charset=utf-8><title>OCR 擂台</title>
<style>body{{font:15px/1.5 system-ui,'Noto Sans JP','Noto Sans TC',sans-serif;margin:16px}}
table{{border-collapse:collapse;width:100%}}td,th{{border:1px solid #ccc;padding:6px;vertical-align:top}}
img{{max-width:220px;max-height:260px}}.pg{{background:#eee;text-align:left}}</style>
<table><tr><th>#</th><th>裁圖</th><th>PaddleOCR-VL-1.5</th><th>manga-ocr</th><th>Qwen3.8</th></tr>
{''.join(rows)}</table>"""
open(f"{OUT_DIR}/report.html", "w").write(page)
from IPython.display import HTML, display
display(HTML(page))
print("已存:", f"{OUT_DIR}/report.html")

## 7. 關閉執行階段

結果已在 Drive 的 `manga-ocr/out/`(`report.html`、`results.json`、裁圖)。

In [ ]:
from google.colab import runtime
runtime.unassign()